# Подготовка набора данных

Подготовка библиотек и отключение предупреждений

In [ ]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

# Импорт библиотеки
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

sns.set_theme(style="ticks")

Подключений Google Drive к блокноту Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Загрузка набора данных

Набор данных, используемый в лабораторных работах доступен по [ссылке](https://www.kaggle.com/datasets/mssmartypants/paris-housing-classification)

Поскольку работы выполнены в среде Google Colaboratory, путь до файла указывается относительно директории, к которой примонтирован Google диск

In [ ]:
ParisHousingClass_df = pd.read_csv('/content/drive/MyDrive/ParisHousingClass.csv')
ParisHousingClass_df.head()

,squareMeters,numberOfRooms,hasYard,hasPool,floors,cityCode,cityPartRange,numPrevOwners,made,isNewBuilt,hasStormProtector,basement,attic,garage,hasStorageRoom,hasGuestRoom,price,category
0,75523,3,0,1,63,9373,3,8,2005,0,1,4313,9005,956,0,7,7559081.5,Basic
1,80771,39,1,1,98,39381,8,6,2015,1,0,3653,2436,128,1,2,8085989.5,Luxury
2,55712,58,0,1,19,34457,6,8,2021,0,0,2937,8852,135,1,9,5574642.1,Basic
3,32316,47,0,0,6,27939,10,4,2012,0,1,659,7141,359,0,3,3232561.2,Basic
4,70429,19,1,1,90,38045,3,7,1990,1,0,8435,2429,292,1,4,7055052.0,Luxury


Разделение на матрицу признаков и зависимую переменную

In [ ]:
X = ParisHousingClass_df.drop(['price'], axis=1)
y = ParisHousingClass_df.price
print("Матрица признаков")
print(X.values)
print("Зависимая переменная")
print(y.values)

Матрица признаков
[[75523 3 0 ... 0 7 'Basic']
 [80771 39 1 ... 1 2 'Luxury']
 [55712 58 0 ... 1 9 'Basic']
 ...
 [83841 3 0 ... 1 9 'Basic']
 [59036 70 0 ... 1 4 'Basic']
 [1440 84 0 ... 1 6 'Basic']]
Зависимая переменная
[7559081.5 8085989.5 5574642.1 ... 8390030.5 5905107.   146708.4]


# Обработка категориальных данных

Замена категории кодом `LabelEncoder`

In [ ]:
from sklearn.preprocessing import LabelEncoder

category_encoder = LabelEncoder()
print("Категориальная переменная до обработки")
print(X.category)
encoded_category = category_encoder.fit_transform(X.category)
print("Категориальная переменная после обработки")
print(encoded_category)

Категориальная переменная до обработки
0        Basic
1       Luxury
2        Basic
3        Basic
4       Luxury
         ...  
9995     Basic
9996     Basic
9997     Basic
9998     Basic
9999     Basic
Name: category, Length: 10000, dtype: object
Категориальная переменная после обработки
[0 1 0 ... 0 0 0]


Метод `OneHotEncoder` используется для преобразования категориальных данных. В используемых данных отсутствуют категориальные переменные, у которых больше двух вариантов, в связи с чем метод не используется.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer


categoricalColumns = ['category', 'numPrevOwners']
droppedColumns = ['cityCode']
leaveColumns = X.columns.difference(categoricalColumns)
leaveColumns = leaveColumns.difference(droppedColumns)

# Создаем список трансформеров
transformers = [
    ('text_encoder', OrdinalEncoder(), categoricalColumns),
    ('preprocessed', 'passthrough', leaveColumns)
]
# Создаем объект ColumnTransformer и передаем ему список трансформеров
data_preprocessor = ColumnTransformer(transformers)

# Линейная регрессия

Разделим выборку на тестовую и тренировочную и обучим предсказывать стоимость собственности с использованием линейного регрессора

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 17)
regression_pipeline = make_pipeline(data_preprocessor, LinearRegression())

In [ ]:
regression_pipeline.fit(X_train, y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('text_encoder',
                                                  OrdinalEncoder(),
                                                  ['category',
                                                   'numPrevOwners']),
                                                 ('preprocessed', 'passthrough',
                                                  Index(['attic', 'basement', 'cityPartRange', 'floors', 'garage',
       'hasGuestRoom', 'hasPool', 'hasStorageRoom', 'hasStormProtector',
       'hasYard', 'isNewBuilt', 'made', 'numberOfRooms', 'squareMeters'],
      dtype='object'))])),
                ('linearregression', LinearRegression())])

## Обработка результатов, тюнинг модели

Предсказание результатов

In [ ]:
y_pred = regression_pipeline.predict(X_test)
print(y_pred)

[7368649.07268037 6113691.76919987 6787416.08398009 ... 4271800.55249771
 8531375.85334669 3508896.22838071]


Оптимизация модели

In [ ]:
X_fic = data_preprocessor.transform(X)
X_fic = np.append(np.ones((X_fic.shape[0], 1), dtype='int'), values=X_fic, axis=1)
X_fic.shape

(10000, 17)

In [ ]:
import statsmodels.api as sm


def retrain(feats_indices):
    X_opt = X_fic[:, feats_indices]
    y_opt = y.to_numpy()
    regressor_OLS = sm.OLS(endog=y_opt, exog=X_opt).fit()
    return regressor_OLS.summary(slim=True)

`Backward elimination` представляет собой метод отбора признаков в моделях линейной регрессии. Исходная модель содержит все признаки, которые поочередно следует исключать с целью найти «худшие» и не применять их в модели.

Исключаются те строки, где параметр `P>|t|` наибольший

In [ ]:
used_columns = [*range(X_fic.shape[1])]
retrain(used_columns)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
No. Observations:               10000   F-statistic:                 1.435e+09
Covariance Type:            nonrobust   Prob (F-statistic):               0.00
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4918.4089   4092.917      1.202      0.230   -3104.534    1.29e+04
x1           -25.3716     75.657     -0.335      0.737    -173.674     122.931
x2            -0.3463      6.655     -0.052      0.958     -13.391      12.699
x3            -0.0045      0.007     -0.688      0.492      -0.017       0.008
x4            -0.0021      0.007     -0.322      0.748      -0.015       0.011
x5            47.2491      6.614      7.143      0.000      34.284      60.215
x6            54.5396      0.658     82.933      0.000      53.251      55.829
x7             0.1135      0.073      1.565      0.118      -0.029       0.256
x8            -5.6042      5.984     -0.936      0.349     -17.335       6.127
x9          2983.1466     42.466     70.248      0.000    2899.904    3066.389
x10           19.3816     38.026      0.510      0.610     -55.156      93.919
x11          141.4944     37.978      3.726      0.000      67.049     215.939
x12         3018.2498     42.338     71.289      0.000    2935.258    3101.241
x13          164.5297     42.637      3.859      0.000      80.953     248.106
x14           -2.3319      2.040     -1.143      0.253      -6.331       1.668
x15            0.2442      0.660      0.370      0.711      -1.049       1.537
x16          100.0000      0.001   1.51e+05      0.000      99.999     100.001
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.25e+07. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
used_columns.pop(2)
retrain(used_columns)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
No. Observations:               10000   F-statistic:                 1.531e+09
Covariance Type:            nonrobust   Prob (F-statistic):               0.00
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4918.4533   4092.713      1.202      0.229   -3104.088    1.29e+04
x1           -25.3452     75.651     -0.335      0.738    -173.637     122.947
x2            -0.0045      0.007     -0.688      0.492      -0.017       0.008
x3            -0.0021      0.007     -0.322      0.748      -0.015       0.011
x4            47.2459      6.614      7.144      0.000      34.282      60.210
x5            54.5395      0.658     82.937      0.000      53.251      55.829
x6             0.1135      0.073      1.564      0.118      -0.029       0.256
x7            -5.6025      5.984     -0.936      0.349     -17.333       6.128
x8          2983.1546     42.464     70.252      0.000    2899.917    3066.392
x9            19.3190     38.005      0.508      0.611     -55.178      93.816
x10          141.4897     37.976      3.726      0.000      67.049     215.931
x11         3018.2337     42.335     71.294      0.000    2935.249    3101.219
x12          164.5573     42.631      3.860      0.000      80.991     248.123
x13           -2.3326      2.040     -1.143      0.253      -6.332       1.667
x14            0.2436      0.660      0.369      0.712      -1.049       1.537
x15          100.0000      0.001   1.51e+05      0.000      99.999     100.001
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.25e+07. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
used_columns.pop(3)
retrain(used_columns)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
No. Observations:               10000   F-statistic:                 1.641e+09
Covariance Type:            nonrobust   Prob (F-statistic):               0.00
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4899.6411   4092.112      1.197      0.231   -3121.723    1.29e+04
x1           -25.3707     75.648     -0.335      0.737    -173.656     122.915
x2            -0.0045      0.007     -0.687      0.492      -0.017       0.008
x3            47.2355      6.613      7.142      0.000      34.272      60.199
x4            54.5382      0.658     82.940      0.000      53.249      55.827
x5             0.1135      0.073      1.564      0.118      -0.029       0.256
x6            -5.5631      5.983     -0.930      0.352     -17.290       6.164
x7          2983.2459     42.461     70.259      0.000    2900.014    3066.478
x8            19.4484     38.001      0.512      0.609     -55.041      93.938
x9           141.5525     37.974      3.728      0.000      67.116     215.989
x10         3018.3509     42.331     71.303      0.000    2935.373    3101.329
x11          164.7546     42.625      3.865      0.000      81.201     248.308
x12           -2.3289      2.040     -1.142      0.254      -6.328       1.670
x13            0.2467      0.659      0.374      0.708      -1.046       1.539
x14          100.0000      0.001   1.52e+05      0.000      99.999     100.001
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.25e+07. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
used_columns.pop(1)
retrain(used_columns)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
No. Observations:               10000   F-statistic:                 1.767e+09
Covariance Type:            nonrobust   Prob (F-statistic):               0.00
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4914.1711   4091.700      1.201      0.230   -3106.386    1.29e+04
x1            -0.0045      0.007     -0.683      0.494      -0.017       0.008
x2            47.2642      6.613      7.148      0.000      34.302      60.226
x3            54.5391      0.658     82.946      0.000      53.250      55.828
x4             0.1136      0.073      1.566      0.117      -0.029       0.256
x5            -5.5563      5.982     -0.929      0.353     -17.283       6.170
x6          2976.8833     37.985     78.370      0.000    2902.425    3051.342
x7            19.4035     37.999      0.511      0.610     -55.082      93.889
x8           141.4822     37.972      3.726      0.000      67.050     215.915
x9          3012.0864     37.986     79.295      0.000    2937.627    3086.546
x10          158.2706     37.986      4.166      0.000      83.809     232.732
x11           -2.3332      2.040     -1.144      0.253      -6.332       1.666
x12            0.2471      0.659      0.375      0.708      -1.046       1.540
x13          100.0000      0.001   1.52e+05      0.000      99.999     100.001
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.25e+07. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
used_columns.pop(12)
retrain(used_columns)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
No. Observations:               10000   F-statistic:                 1.915e+09
Covariance Type:            nonrobust   Prob (F-statistic):               0.00
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4920.5222   4091.489      1.203      0.229   -3099.621    1.29e+04
x1            -0.0045      0.007     -0.679      0.497      -0.017       0.008
x2            47.2838      6.612      7.151      0.000      34.323      60.245
x3            54.5445      0.657     82.977      0.000      53.256      55.833
x4             0.1142      0.073      1.575      0.115      -0.028       0.256
x5            -5.5886      5.981     -0.934      0.350     -17.313       6.136
x6          2977.1292     37.978     78.391      0.000    2902.685    3051.574
x7            19.3277     37.997      0.509      0.611     -55.153      93.809
x8           141.4570     37.970      3.725      0.000      67.028     215.886
x9          3011.9223     37.982     79.300      0.000    2937.471    3086.374
x10          158.2298     37.985      4.166      0.000      83.772     232.687
x11           -2.3305      2.040     -1.142      0.253      -6.329       1.668
x12          100.0000      0.001   1.52e+05      0.000      99.999     100.001
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.25e+07. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
used_columns.pop(7)
retrain(used_columns)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
No. Observations:               10000   F-statistic:                 2.089e+09
Covariance Type:            nonrobust   Prob (F-statistic):               0.00
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4945.9621   4091.032      1.209      0.227   -3073.284     1.3e+04
x1            -0.0045      0.007     -0.679      0.497      -0.017       0.008
x2            47.2466      6.611      7.146      0.000      34.287      60.206
x3            54.5457      0.657     82.983      0.000      53.257      55.834
x4             0.1154      0.072      1.593      0.111      -0.027       0.257
x5            -5.5414      5.980     -0.927      0.354     -17.264       6.181
x6          2977.1557     37.976     78.395      0.000    2902.714    3051.597
x7           141.4839     37.969      3.726      0.000      67.058     215.910
x8          3011.7456     37.979     79.301      0.000    2937.300    3086.191
x9           158.3548     37.982      4.169      0.000      83.901     232.808
x10           -2.3387      2.040     -1.147      0.252      -6.337       1.660
x11          100.0000      0.001   1.52e+05      0.000      99.999     100.001
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.25e+07. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
used_columns.pop(1)
retrain(used_columns)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
No. Observations:               10000   F-statistic:                 2.298e+09
Covariance Type:            nonrobust   Prob (F-statistic):               0.00
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4961.5833   4090.857      1.213      0.225   -3057.320     1.3e+04
x1            47.1984      6.611      7.140      0.000      34.240      60.157
x2            54.5460      0.657     82.985      0.000      53.258      55.834
x3             0.1155      0.072      1.593      0.111      -0.027       0.258
x4            -5.4914      5.980     -0.918      0.358     -17.213       6.230
x5          2977.4661     37.973     78.411      0.000    2903.032    3051.900
x6           141.5866     37.967      3.729      0.000      67.163     216.010
x7          3011.8215     37.977     79.306      0.000    2937.378    3086.265
x8           157.8284     37.974      4.156      0.000      83.393     232.264
x9            -2.3577      2.040     -1.156      0.248      -6.356       1.640
x10          100.0000      0.001   1.52e+05      0.000      99.999     100.001
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.24e+07. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
used_columns.pop(4)
retrain(used_columns)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
No. Observations:               10000   F-statistic:                 2.553e+09
Covariance Type:            nonrobust   Prob (F-statistic):               0.00
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       4913.4536   4090.489      1.201      0.230   -3104.729    1.29e+04
x1            47.2418      6.611      7.146      0.000      34.284      60.200
x2            54.5587      0.657     83.024      0.000      53.271      55.847
x3             0.1166      0.072      1.609      0.108      -0.025       0.259
x4          2977.4196     37.972     78.410      0.000    2902.986    3051.853
x5           141.8218     37.966      3.735      0.000      67.401     216.243
x6          3012.0739     37.976     79.315      0.000    2937.633    3086.515
x7           157.1333     37.966      4.139      0.000      82.713     231.554
x8            -2.3481      2.039     -1.151      0.250      -6.346       1.650
x9           100.0000      0.001   1.52e+05      0.000      99.999     100.001
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.24e+07. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
used_columns.pop(8)
retrain(used_columns)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
No. Observations:               10000   F-statistic:                 2.872e+09
Covariance Type:            nonrobust   Prob (F-statistic):               0.00
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        204.9843     83.147      2.465      0.014      41.999     367.970
x1            47.1824      6.611      7.137      0.000      34.224      60.140
x2            54.5549      0.657     83.018      0.000      53.267      55.843
x3             0.1161      0.072      1.603      0.109      -0.026       0.258
x4          2977.3453     37.973     78.407      0.000    2902.911    3051.780
x5           141.8475     37.967      3.736      0.000      67.425     216.270
x6          3011.9816     37.977     79.311      0.000    2937.540    3086.424
x7           157.2097     37.966      4.141      0.000      82.788     231.631
x8           100.0000      0.001   1.52e+05      0.000      99.999     100.001
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.59e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [ ]:
used_columns.pop(3)
retrain(used_columns)

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       1.000
Model:                            OLS   Adj. R-squared:                  1.000
No. Observations:               10000   F-statistic:                 3.282e+09
Covariance Type:            nonrobust   Prob (F-statistic):               0.00
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const        269.4232     72.789      3.701      0.000     126.742     412.104
x1            47.1664      6.611      7.135      0.000      34.207      60.125
x2            54.5669      0.657     83.035      0.000      53.279      55.855
x3          2977.6423     37.975     78.410      0.000    2903.203    3052.082
x4           142.0881     37.969      3.742      0.000      67.660     216.516
x5          3011.6926     37.979     79.298      0.000    2937.246    3086.140
x6           157.3608     37.969      4.144      0.000      82.934     231.788
x7           100.0000      0.001   1.52e+05      0.000      99.999     100.001
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 2.31e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

Для проверки результатов выведем колонки со значениеми `P`, близкими к нулю

In [ ]:
print('Оставшиеся колонки со значением P, близким к нулю')
columns_left = used_columns[1:]
columns_left = [x - 1 for x in columns_left]
columns_left

Оставшиеся колонки со значением P, близким к нулю


[4, 5, 8, 10, 11, 12, 15]

In [ ]:
print('Названия оставшихся колонок')
expected_columns = categoricalColumns + leaveColumns.to_list()
[expected_columns[x] for x in columns_left]

Названия оставшихся колонок


['cityPartRange',
 'floors',
 'hasPool',
 'hasStormProtector',
 'hasYard',
 'isNewBuilt',
 'squareMeters']

In [ ]:
column_dropper = ColumnTransformer(transformers=[
    ('leave_best_columns', 'passthrough', columns_left)
])

In [ ]:
regression_pipeline = make_pipeline(data_preprocessor, column_dropper, LinearRegression())
regression_pipeline.fit(X_train, y_train)

Pipeline(steps=[('columntransformer-1',
                 ColumnTransformer(transformers=[('text_encoder',
                                                  OrdinalEncoder(),
                                                  ['category',
                                                   'numPrevOwners']),
                                                 ('preprocessed', 'passthrough',
                                                  Index(['attic', 'basement', 'cityPartRange', 'floors', 'garage',
       'hasGuestRoom', 'hasPool', 'hasStorageRoom', 'hasStormProtector',
       'hasYard', 'isNewBuilt', 'made', 'numberOfRooms', 'squareMeters'],
      dtype='object'))])),
                ('columntransformer-2',
                 ColumnTransformer(transformers=[('leave_best_columns',
                                                  'passthrough',
                                                  [4, 5, 8, 10, 11, 12, 15])])),
                ('linearregression', LinearRegression())])

Найдем коэффициент детерминации, который поможет измерить качество модели.

In [ ]:
from sklearn.metrics import r2_score


score = r2_score(y_test, regression_pipeline.predict(X_test))
print(f'Оценка R^2 (коэффициент детерминации) на тестовой выборке: {score:.3f}')

Оценка R^2 (коэффициент детерминации) на тестовой выборке: 1.000


`R2 = 1`, что указывает на высокое качество модели

Это свидетельствует о том, что оставшиеся переменные являются значимыми для прогнозирования